# Uncertainty-Aware Jacobi Set Computation

Demo of the uncertainty-aware Jacobi set computation on the analytic dataset of the paper (Sec. 5.1).
All figures are written to `figures/`.

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from uncertainty_aware_jacobi_set import FieldPair, data, geometry, jacobi, models, vis

FIG = Path("figures")
FIG.mkdir(exist_ok=True)
plt.rcParams["figure.dpi"] = 110
plt.rcParams.update({"pdf.fonttype": 42, "ps.fonttype": 42})  # editable text in the PDFs


def save(fig, name, png=False):
    fig.savefig(FIG / f"{name}.pdf", bbox_inches="tight", pad_inches=0.02)
    if png:
        fig.savefig(FIG / f"{name}.png", dpi=150, bbox_inches="tight", pad_inches=0.02)

## 1. Analytic dataset

Two closed-form scalar fields on a $50 \times 50$ grid, whose exact Jacobi set $\{\kappa = 0\}$ is known analytically.

In [ ]:
RES = 50
d = data.analytic_grid(RES)
geo = geometry.build_geometry(d["x1d"], d["y1d"])
mean_f, mean_g = d["f"].ravel(), d["g"].ravel()
print(f"{len(geo['P'])} vertices, {len(geo['tris'])} triangles, {len(geo['T1'])} interior edges")

fine = data.analytic_grid(4 * RES)


def exact_jacobi_set(ax):
    ax.contour(fine["x1d"], fine["y1d"], fine["kappa_exact"], levels=[0.0],
               colors="k", linewidths=1.0, alpha=0.85)


fig, ax = plt.subplots(1, 3, figsize=(15, 4.4))
vis.scalar_field(geo, mean_f, ax=ax[0], cbar=True, title=r"$\mu_f$")
vis.scalar_field(geo, mean_g, ax=ax[1], cbar=True, title=r"$\mu_g$")
exact_jacobi_set(ax[2])
ax[2].set_aspect("equal")
ax[2].set_title(r"exact Jacobi set $\{\kappa = 0\}$")
plt.tight_layout()
save(fig, "analytic_dataset")
plt.show()

## 2. Uncertainty model

A squared exponential kernel per field ($\ell = 0.15$, $\Sigma_{fg} = 0$).
The standard deviation is zero below an onset $y$-value and increases linearly to 1% of the field's standard deviation at $y = 1$.

In [ ]:
ELL = 0.15                  # length scale of the kernel
REL_SIGMA = 0.01            # standard deviation at y = 1, relative to the field's std
ONSETS = [0.0, -0.5, -1.0]  # sigma = 0 below this y-value, linear ramp above

P = geo["P"]
pairs, sigmas_f = {}, {}
for y0 in ONSETS:
    ramp = np.clip((P[:, 1] - y0) / (1.0 - y0), 0.0, 1.0)
    sigmas_f[y0] = REL_SIGMA * mean_f.std() * ramp
    L_f = models.kernel_factor(P, sigmas_f[y0], ELL)
    L_g = models.kernel_factor(P, REL_SIGMA * mean_g.std() * ramp, ELL)
    pairs[y0] = FieldPair.independent(mean_f, L_f, mean_g, L_g)

fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))
for a, y0 in zip(ax, ONSETS):
    vis.scalar_field(geo, sigmas_f[y0], ax=a, cmap="magma", cbar=True,
                     title=rf"$\sigma_i$ of $f$, onset $y = {y0:g}$")
plt.tight_layout()
save(fig, "uncertainty_model")
plt.show()

## 3. Jacobi set edge probability

`jacobi.compute` (Algorithm 1) yields the Jacobi set edge probability $p_e$, shown for the three onsets with the exact Jacobi set in black (cf. Fig. 4).

In [ ]:
results = {y0: jacobi.compute(geo, pairs[y0]) for y0 in ONSETS}

fig, ax = plt.subplots(1, 3, figsize=(15, 4.6))
for a, y0 in zip(ax, ONSETS):
    vis.edge_probability(geo, results[y0]["p_edge"], ax=a,
                         title=f"onset y = {y0:g}", export=True)
    exact_jacobi_set(a)
plt.tight_layout()
save(fig, "edge_probability", png=True)
plt.show()

## 4. Expected Jacobi set degree

The aggregation of $p_e$ to the vertices, $\mathrm{E}(D_i)$ (left), and its signed variant (right), with the Jacobi set of the mean fields in orange.

In [ ]:
Y0 = -0.5
pair, res = pairs[Y0], results[Y0]

fig, ax = plt.subplots(1, 2, figsize=(13, 5.4))
vis.expected_degree(geo, res["expected_degree"], ax=ax[0],
                    title=r"expected Jacobi set degree $\mathrm{E}(D_i)$")
vis.signed_expected_degree(geo, res["signed_degree"], ax=ax[1],
                           title=r"signed expected Jacobi set degree $\tilde{D}_i$")
vis.jacobi_set(geo, pair.mean_f, pair.mean_g, ax=ax[1])
plt.tight_layout()
save(fig, "expected_degree_signed")
plt.show()

## 5. Layered visualization

Expected Jacobi set degree, Jacobi set of the mean fields (orange), and three Jacobi set samples.
Right: the same with the optional local bilinear lines (RCLB).

In [ ]:
fs, gs = pair.sample(3, np.random.default_rng(0))

fig, ax = plt.subplots(1, 2, figsize=(13, 5.4))
for a, method in zip(ax, ["pl", "rclb"]):
    vis.expected_degree(geo, res["expected_degree"], ax=a, title=f"{method.upper()} lines")
    for i in range(3):
        vis.jacobi_set(geo, fs[i], gs[i], ax=a, color=vis.SAMPLE_COLORS[i], lw=0.9,
                       method=method, label=f"Jacobi set sample {i + 1}")
    vis.jacobi_set(geo, pair.mean_f, pair.mean_g, ax=a, lw=1.1, method=method,
                   label="mean Jacobi set")
plt.tight_layout()
fig.legend(*ax[0].get_legend_handles_labels(), loc="upper center",
           bbox_to_anchor=(0.5, 0.0), ncol=4, frameon=False)
save(fig, "layered_visualization", png=True)
plt.show()

## 6. Ensemble data

The uncertainty model can also be estimated from ensemble members.
As a stand-in for real ensemble data, we draw 10 members from the model with onset $y = -1$.

In [ ]:
members_f, members_g = pairs[-1.0].sample(10, np.random.default_rng(1))  # (E, N) each
ens = FieldPair.from_ensemble(members_f, members_g)
res_ens = jacobi.compute(geo, ens)
print(f"factor L: {ens.L.shape[0]} x {ens.L.shape[1]}")

fs, gs = ens.sample(3, np.random.default_rng(0))
fig, ax = plt.subplots(figsize=(6.5, 5.4))
vis.expected_degree(geo, res_ens["expected_degree"], ax=ax, title="ensemble model, E = 10")
for i in range(3):
    vis.jacobi_set(geo, fs[i], gs[i], ax=ax, color=vis.SAMPLE_COLORS[i], lw=0.9,
                   label=f"Jacobi set sample {i + 1}")
vis.jacobi_set(geo, ens.mean_f, ens.mean_g, ax=ax, lw=1.1, label="mean Jacobi set")
plt.tight_layout()
fig.legend(*ax.get_legend_handles_labels(), loc="upper center",
           bbox_to_anchor=(0.5, 0.0), ncol=2, frameon=False)
save(fig, "ensemble")
plt.show()